# Experiment — Test and Revise a Python Function

A code-correcting feedback loop compares a function's behaviour with test inputs and expected results, returns the observed failures to a model and checks its proposed revision. The requirements, current code and feedback stay connected. More attempts do not guarantee improvement, so the application needs explicit acceptance criteria and an attempt limit.

You will use fictional temperature readings, first test ordinary inputs, then add missing and empty readings, inspect one revision and follow a maximum of three attempts. You will record test outcomes and examine a deliberately non-improving sequence. Everything is supplied and runs independently. Offline mode runs reviewed functions with author-written candidate code; it is not a live model evaluation. Optional OpenAI mode requests fresh code but pauses when code has not been reviewed. No generated text is automatically executed.


## 1. Define the task and predict

`average_reading(values)` initially averages a non-empty list of finite numbers. Predict the result for `[18, 20, 22]`. Later the requirements add ignoring `None` and returning `None` for empty/all-missing inputs. Negative temperatures remain valid. Inputs contain finite numbers and `None` only; arbitrary text and infinity are outside this task.

Adding a new requirement is not proof of an earlier defect. Keep the ordinary tests when adding edge cases.


In [ ]:
task = 'Write average_reading(values), returning the average of the readings.\nThe initial inputs are non-empty lists of finite numbers.\nReturn only the Python function, without Markdown fences.'
expanded_task = 'Write average_reading(values). Inputs contain finite numbers and None only.\nIgnore None values. Return None if no readings remain.\nOtherwise return the average, preserving valid negative temperatures.\nReturn only the Python function, without Markdown fences.'
code_instructions = "Write a simple, readable Python function."
initial_cases = [([18, 20, 22], 20.0)]
test_cases = [([18, 20, 22], 20.0), ([18, None, 22], 20.0), ([], None), ([None], None), ([-4, 0, 4], 0.0)]


## Configuration and the shared helper

The following cell repeats the **unchanged** `get_completion(system_prompt, user_prompt)` adapter so this notebook works independently. Offline mode uses exact prompt/input matches and labels its supplied responses. An edited prompt has no new offline model response. For fresh responses, install the companion requirements in a standard local Python/Jupyter environment; keep `.env` beside `COMPANION_VERSION` at the extracted package root (or beside this notebook when used alone). Reuse the configuration from Section 2.3. Browser Python stays offline. The live client has a request timeout and no automatic retries.


In [ ]:
# Shared offline/live request adapter. No API key is stored in this notebook.
# Code origin: get_completion(system_prompt, user_prompt) was introduced in Section 2.3.
# Notebook extension used from Section 3.2: supplied/live mode and interaction records.
# Evaluation notebooks reuse this helper without changing its two-argument interface.
import json
import os
import sys

MODE = "offline"
MODEL = ""
_client = None
request_records = []
OFFLINE_SAMPLES = {}

# Match supplied examples only when both messages are unchanged.
def _request_key(system_prompt, user_prompt):
    return json.dumps([system_prompt.strip(), user_prompt.strip()], ensure_ascii=False)

# Register a supplied teaching response; this does not contact OpenAI.
def register_sample(system_prompt, user_prompt, response):
    OFFLINE_SAMPLES[_request_key(system_prompt, user_prompt)] = response

# Locate one package-root configuration, while retaining standalone notebook use.
def _configuration_file():
    from pathlib import Path
    bases = [Path.cwd()]
    if "__file__" in globals():
        bases.insert(0, Path(__file__).resolve().parent)
    for base in bases:
        for folder in [base, *base.parents]:
            if (folder / "COMPANION_VERSION").is_file():
                return folder / ".env"
    # Standalone notebooks keep their configuration beside the notebook.
    return bases[0] / ".env"

# Load the private configuration and prepare an OpenAI client when live mode is chosen.
def configure_live(model=""):
    global MODE, MODEL, _client
    if sys.platform in {"emscripten", "wasi"}:
        raise RuntimeError("The OpenAI SDK needs a standard Python kernel. Keep offline mode in browser Python.")
    try:
        from openai import OpenAI
        from dotenv import load_dotenv
    except ImportError:
        raise RuntimeError("Install openai and python-dotenv in your Python kernel, then rerun setup.") from None
    # Read the key and model from the local file without printing the key.
    load_dotenv(_configuration_file())
    selected_model = model.strip() or os.getenv("OPENAI_MODEL", "").strip()
    if not selected_model or selected_model.startswith("YOUR_"):
        raise ValueError("Set OPENAI_MODEL in your .env file before choosing live mode.")
    key = os.getenv("OPENAI_API_KEY", "").strip()
    if not key or key.startswith("YOUR_"):
        raise ValueError("Copy your own OpenAI key into the .env file before choosing live mode.")
    _client = OpenAI(timeout=45.0, max_retries=0)
    MODEL = selected_model
    MODE = "live"
    print("OpenAI configured from .env. No key is printed.")


def get_completion(system_prompt, user_prompt):
    if MODE == "live":
        if _client is None:
            raise RuntimeError("Run configure_live(MODEL) in a private Python kernel first.")
        try:
            # Send role instructions separately from the user task and data.
            response = _client.responses.create(model=MODEL, instructions=system_prompt, input=user_prompt, max_output_tokens=1000, store=False)
        except Exception:
            raise RuntimeError("API request failed. Check your model, account access and connection; no key has been printed.") from None
        text = response.output_text
        print("[Live OpenAI response]")
    else:
        # Offline mode returns stored text, not a newly generated model answer.
        text = OFFLINE_SAMPLES.get(_request_key(system_prompt, user_prompt))
        if text is None:
            print("[No saved response for these edited instructions]")
            return "No model was called. Use OpenAI mode in a standard Python kernel, or label a response you write yourself."
        print("[Supplied teaching response — no API call]")
    # Keep messages and responses for evaluation; never record the API key.
    request_records.append({"mode": MODE, "model": MODEL if MODE == "live" else "supplied-example", "system_prompt": system_prompt, "user_prompt": user_prompt, "response": text})
    return text

# Release the API client after finishing the experiment.
def clear_live():
    global _client, MODE
    if _client is not None:
        _client.close()
    _client = None
    MODE = "offline"
    print("Live client cleared from this kernel.")

In [ ]:
USE_OPENAI = False  # Keep False for browser/offline practice.
if USE_OPENAI:
    configure_live()  # Uses your private local .env; never put the key in a cell.
else:
    print("OFFLINE: supplied examples; not fresh model responses.")


## 2. Inspect the supplied test runner

`run_tests` accepts a reviewed Python function, not model text. It reports input, expected value, actual result or error and pass/fail for every case. Numerical answers use a small tolerance rather than exact floating-point equality. Read the loop and predict which failures the initial function will produce on the expanded cases.

For syntax errors or a missing function, report the error type/message and relevant traceback during review instead of calling an untested attempt successful. The small runner here handles runtime errors inside its supplied functions.


In [ ]:
from math import isclose
from time import perf_counter


def run_tests(function, cases):
    # Each case records its own result. Exceptions are failures, not missing tests.
    result = {"passed": 0, "failed": 0, "cases": []}
    for values, expected in cases:
        row = {"input": values, "expected": expected}
        try:
            actual = function(list(values))
            passed = (actual is None) if expected is None else (
                isinstance(actual, (int, float)) and not isinstance(actual, bool)
                and isclose(actual, expected, rel_tol=1e-9, abs_tol=1e-9))
            row.update(actual=actual, passed=passed)
        except Exception as error:
            row.update(error=type(error).__name__ + ": " + str(error), passed=False)
        result["passed" if row["passed"] else "failed"] += 1
        result["cases"].append(row)
    return result


def format_feedback(result):
    lines = [f"Tests: {result['passed']} passed; {result['failed']} failed."]
    for row in result["cases"]:
        if not row["passed"]:
            actual = row.get("error", repr(row.get("actual")))
            lines.append(f"Input: {row['input']} | Expected: {row['expected']} | Actual/error: {actual}")
    return "\n".join(lines)


def make_revision_prompt(current_code, feedback):
    return f"""
Requirements: {expanded_task}
Current code:
{current_code}
Test feedback:
{feedback}
Fix the reported problems while preserving the other requirements.
Return only the revised function, without Markdown fences.
"""


## 3. Read the reviewed candidate versions

These three short versions are fixed teaching resources. Read them as ordinary Python definitions. The mapping connects each exact supplied code string to the reviewed function that implements it; it does not compile or execute a new string from a model. This boundary is not a sandbox.

Predict what each version fixes and what it still fails. Do not skip directly to the final version during your prediction.


In [ ]:
initial_code = 'def average_reading(values):\n    return sum(values) / len(values)'
partial_code = 'def average_reading(values):\n    valid = [value for value in values if value is not None]\n    return sum(valid) / len(valid)'
complete_code = 'def average_reading(values):\n    valid = [value for value in values if value is not None]\n    if not valid:\n        return None\n    return sum(valid) / len(valid)'

# Version 1 handles ordinary inputs only.
def ordinary_average(values):
    return sum(values) / len(values)

# Version 2 ignores missing readings but still fails on an empty result.
def ignore_missing_average(values):
    valid = [value for value in values if value is not None]
    return sum(valid) / len(valid)

# Version 3 adds the required empty-input behaviour.
def complete_average(values):
    valid = [value for value in values if value is not None]
    if not valid:
        return None
    return sum(valid) / len(valid)

reviewed_candidates = {initial_code: ordinary_average,
                       partial_code: ignore_missing_average,
                       complete_code: complete_average}


## 4. Initial generation, then expanded tests

The same initial request has a supplied offline response and an optional fresh OpenAI response. A live response not matching a reviewed candidate is shown for review; the loop does not execute it automatically. Start with the original tests, then inspect the expanded test feedback. The failure on an added requirement must be described accurately.


In [ ]:
register_sample(code_instructions, task, initial_code)
current_code = get_completion(code_instructions, task)
print("INITIAL CODE\n", current_code)
function = reviewed_candidates.get(current_code)
if function is not None:
    print("ORIGINAL REQUIREMENTS:", format_feedback(run_tests(function, initial_cases)))
    print("EXPANDED REQUIREMENTS:\n", format_feedback(run_tests(function, test_cases)))
else:
    print("New code needs review before any execution.")


## 5. Turn actual failures into revision prompts

The next cell registers supplied candidate revisions only for these exact prompts and observed test results. It demonstrates how the original requirements, code and feedback are combined. It does not claim that a model produces these particular fixes. Optional live mode ignores these examples and generates fresh text instead.


In [ ]:
# Author-written offline responses, tied to observed feedback on reviewed code.
for before, after in [(initial_code, partial_code), (partial_code, complete_code)]:
    observed = run_tests(reviewed_candidates[before], test_cases)
    prompt = make_revision_prompt(before, format_feedback(observed))
    register_sample(code_instructions, prompt, after)


## 6. Follow the bounded loop

Three attempts include the initial version plus at most two revisions. Passing tests stops revision requests immediately; reaching the limit stops with unresolved work. Unreviewed code stops before execution. The printed counts are observations from the supplied functions, not a prediction about live performance.


In [ ]:
def run_feedback_loop(start_code, cases, max_attempts=3):
    if max_attempts < 1:
        raise ValueError("Allow at least one attempt.")
    current_code = start_code
    history = []
    stop_reason = "attempt limit"
    for attempt in range(1, max_attempts + 1):
        # Only supplied/reviewed functions run automatically. Never exec model text.
        function = reviewed_candidates.get(current_code)
        if function is None:
            print("Review this new code before running it:\n", current_code)
            stop_reason = "unreviewed code"
            break
        started = perf_counter()
        result = run_tests(function, cases)
        feedback = format_feedback(result)
        history.append({"attempt": attempt, "code": current_code,
                        "result": result, "feedback": feedback,
                        "test_seconds": perf_counter() - started})
        print(f"Attempt {attempt}: {result['passed']} passed, {result['failed']} failed")
        if result["failed"] == 0:
            stop_reason = "all supplied tests passed"
            break
        if attempt == max_attempts:
            break  # No unused revision request after the limit.
        revision_prompt = make_revision_prompt(current_code, feedback)
        current_code = get_completion(code_instructions, revision_prompt)
    print("STOP:", stop_reason)
    return history, stop_reason, current_code


In [ ]:
history, stop_reason, final_code = run_feedback_loop(current_code, test_cases, max_attempts=3)
print("Model/helper requests recorded:", len(request_records))


## 7. Inspect the trace and stopping behaviour

Inspect the actual feedback, not just the passed count. Keep the prompt and response records, code, expected/actual values, runtime errors and stopping reason. Tests may regress, and the same bad response may repeat. The optional non-improving experiment below uses a local fake responder so it can never make OpenAI calls.


In [ ]:
for row in history:
    print(f"\nATTEMPT {row['attempt']} — test time {row['test_seconds']:.6f} s")
    print(row["feedback"])
    print("CODE:\n", row["code"])
print("STOPPING REASON:", stop_reason)


In [ ]:
RUN_STAGNATION_DEMO = False
if RUN_STAGNATION_DEMO:
    original_helper = get_completion
    def repeat_same_code(system_prompt, user_prompt):
        print("[Deliberately repeated supplied code — no API call]")
        return initial_code
    get_completion = repeat_same_code
    try:
        stalled_history, stalled_stop, stalled_code = run_feedback_loop(initial_code, test_cases, 3)
    finally:
        get_completion = original_helper
    print("Unresolved failures:", stalled_history[-1]["result"]["failed"])


## 8. Local live review and independent change

To request fresh code, use a standard local Python kernel and your private `.env`, set `USE_OPENAI = True` and run one case. At most three calls occur in this sequence: initial generation and two revisions. The loop stops immediately if a response is not already reviewed; it may therefore use fewer calls. Model output may include code fences despite the instruction; treat that as a format problem to review, not something to execute blindly.

**To test a new candidate locally:** read the displayed function, compare it with the requirements and check that it contains only the intended calculation. Paste the reviewed function into a new Python cell, run that cell, and call `run_tests(average_reading, test_cases)`. If it passes your review, add `reviewed_candidates[final_code] = average_reading` before continuing with `run_feedback_loop(final_code, test_cases, max_attempts=1)`. This tests that reviewed candidate without requesting another revision. Do not change the helper or present this manual approval as automatic verification. Restart/rerun only the necessary cells; preserve the trace and count of requests. Do not make a batch of live calls to explore the examples.

**Change:** add one decimal or all-missing case, predict the result and test the reviewed functions. Keep the existing cases. A new offline prompt may have no registered response: record a proposed test rather than claiming new model behaviour. Do not weaken the tests to make a candidate pass.

**Explain:** which revisions handled missing/empty input? Did they preserve negative readings? Were these code defects or new requirements? What remains untested? Compare with manual debugging. Do the results justify another attempt, or should the unresolved result go to a person?

**Knowledge check:** the revision request needs the full requirements to avoid fixing one failure while breaking another. Three total attempts allow two revisions. At the limit, keep failures marked unresolved. Five passing tests establish only the tested cases, not general correctness.


In [ ]:
clear_live()